# Chapter 4 — Session 2
## Regression • Trend Analysis • EDA • Auto-EDA 


## 0) Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)

print('Setup complete.')


### ✅ Your Turn: Verify Setup
Run the cell above. If you see "Setup complete.", you're ready! 

**Quick check:** What does `plt.rcParams['figure.figsize'] = (7, 4)` do?
- [ ] Sets the default figure size to 7 inches wide × 4 inches tall
- [ ] Creates a new plot
- [ ] Imports a plotting library

## 1) Datasets
We create two simple synthetic datasets: a regression example (study_hours → marks) and a weekly sales trend.

In [ ]:
# Create a random number generator with seed=7 for reproducibility
# (Using the same seed means everyone gets the same "random" data)
rng = np.random.default_rng(7)

# ═══════════════════════════════════════════════════════════════
# REGRESSION DATASET: Simulating study_hours → marks relationship
# ═══════════════════════════════════════════════════════════════
n = 80  # Number of students in our fake dataset

# Generate random study hours: mean=5, std=2, for n students
# .clip(0) ensures no negative values (can't study negative hours!)
study_hours = rng.normal(5, 2, n).clip(0)

# Add random noise to make data realistic (not a perfect line)
# Real-world data always has some randomness/error
noise = rng.normal(0, 8, n)

# Create marks using a linear formula: marks = 12*hours + 20 + noise
# - 12 is the "true" slope (each hour adds ~12 marks)
# - 20 is the "true" intercept (base marks with 0 study)
# .clip(0, 100) keeps marks between 0 and 100 (realistic range)
marks = (12*study_hours + 20 + noise).clip(0, 100)

# Combine into a DataFrame (table format)
df_reg = pd.DataFrame({'study_hours': study_hours, 'marks': marks})

# ═══════════════════════════════════════════════════════════════
# TREND DATASET: Simulating weekly sales with trend + seasonality
# ═══════════════════════════════════════════════════════════════
weeks = np.arange(1, 53)  # Weeks 1 through 52 (one year)

# Linear trend: sales grow by 3 units each week, starting at ~200
trend = 200 + 3*weeks

# Seasonal pattern using sine wave:
# - Amplitude 15 (sales swing ±15 from trend)
# - Period of 12 weeks (pattern repeats every ~3 months)
# - 2*np.pi/12 makes one complete wave every 12 weeks
season = 15*np.sin(2*np.pi*weeks/12)

# Random noise to simulate real-world unpredictability
sales_noise = rng.normal(0, 10, len(weeks))

# Final sales = trend + seasonal pattern + random noise
sales = trend + season + sales_noise

# Combine into a DataFrame
df_trend = pd.DataFrame({'week': weeks, 'sales': sales})

print('df_reg and df_trend created.')

### Quick peek

In [ ]:
df_reg.head()

In [ ]:
df_trend.head()

### 🧪 Explore the Data (Your Turn)
Before we dive into regression, let's explore the datasets. Complete the exercises below:

In [ ]:
# TODO: Exercise 1 - Find basic statistics for df_reg
# Hint: Use .describe() method

# Your code here:


# TODO: Exercise 2 - What is the correlation between study_hours and marks?
# Hint: Use df_reg['study_hours'].corr(df_reg['marks'])

# Your code here:


# TODO: Exercise 3 - How many weeks of sales data do we have?
# Hint: Use len() or .shape

# Your code here:


---
## 2) Simple Linear Regression
We model `marks ≈ m * study_hours + c`.

- `m` is the slope (average change in marks per extra study hour).
- `c` is the intercept (predicted marks when study_hours = 0).

We will plot the data, fit a line with `np.polyfit`, and compute R² manually.

In [ ]:
plt.scatter(df_reg['study_hours'], df_reg['marks'])
plt.xlabel('study_hours')
plt.ylabel('marks')
plt.title('Scatter: study_hours vs marks')
plt.show()


**🤔 Think About It:** Looking at the scatter plot above, do you see a pattern? 
- Does it look like there's a relationship between study hours and marks?
- Is the relationship positive (more study → higher marks) or negative?
- Are there any outliers (unusual points)?

*Write your observations in a markdown cell below or discuss with a partner.*

### Fit line with `np.polyfit` (simple and explainable)
`np.polyfit(x, y, 1)` returns slope and intercept for the best-fit line (degree 1 polynomial).

In [ ]:
x = df_reg['study_hours'].to_numpy()
y = df_reg['marks'].to_numpy()

m, c = np.polyfit(x, y, 1)
print('slope m =', round(m,3), ' intercept c =', round(c,3))

# Plot with fitted line
x_line = np.linspace(x.min(), x.max(), 100)
y_line = m * x_line + c
plt.scatter(x, y)
plt.plot(x_line, y_line, color='orange')
plt.xlabel('study_hours')
plt.ylabel('marks')
plt.title('Regression line (np.polyfit)')
plt.show()


### Compute R² manually (intuitive)
- SSE = sum((y - y_pred)^2)
- SST = sum((y - mean(y))^2)
- R² = 1 - SSE/SST (fraction of variance explained)

In [ ]:
y_pred = m*x + c
sse = np.sum((y - y_pred)**2)
stt = np.sum((y - y.mean())**2)
r2 = 1 - sse/stt
print('R² =', round(r2,3))


### 🧪 Try It Yourself: Predict Marks
Use the slope and intercept we just calculated to make predictions!

In [ ]:
# TODO: Use the regression equation to predict marks for different study hours
# Formula: predicted_marks = m * study_hours + c

# Example: Predict marks for a student who studies 3 hours
study_hours_new = 3
predicted_marks = ___  # Fill in the formula using m, c, and study_hours_new
print(f"Predicted marks for {study_hours_new} hours of study: {predicted_marks:.1f}")

# TODO: Now predict for 8 hours of study
study_hours_new = 8
predicted_marks = ___  # Your code here
print(f"Predicted marks for {study_hours_new} hours of study: {predicted_marks:.1f}")

# 🤔 Question: What happens if you predict for 0 hours? Does the result make sense?
# Try it below:


### Residuals (diagnostic)
Residual = actual - predicted. Plot residuals vs X; patterns signal problems (curve → non-linearity, funnel → heteroskedasticity).

In [ ]:
residuals = y - y_pred
plt.scatter(x, residuals)
plt.axhline(0, linestyle='--', color='gray')
plt.xlabel('study_hours')
plt.ylabel('residual (actual - predicted)')
plt.title('Residuals vs study_hours')
plt.show()


**🎯 Interpreting Residuals:** 
- If residuals scatter randomly around 0 → linear model is appropriate ✅
- If residuals show a curve → try a non-linear model ⚠️
- If residuals fan out (funnel shape) → heteroskedasticity (variance not constant) ⚠️

**What do you observe in the residual plot above?**

### ✍️ Exercise: Interpret the Slope
In one sentence, explain what the slope means here and one limitation of this interpretation.

**Your answer:** *(double-click to edit)*

- The slope means: ___
- One limitation: ___

---
## 3) Trend Analysis (Time Series)
Plot weekly sales and smooth with a moving average (simple repeated averaging).

In [ ]:
plt.plot(df_trend['week'], df_trend['sales'], marker='o')
plt.xlabel('week')
plt.ylabel('sales')
plt.title('Weekly sales (raw)')
plt.show()


**🤔 Observation Questions:**
1. Do you see an overall upward or downward trend?
2. Are there any repeating patterns (seasonality)?
3. Are there any unusual spikes or dips?

### Smooth with moving average (`.rolling(window).mean()`)
A moving average replaces each point with the average of nearby points; window size controls smoothing strength.

In [ ]:
df_trend['sales_ma4'] = df_trend['sales'].rolling(window=4).mean()
plt.plot(df_trend['week'], df_trend['sales'], label='sales')
plt.plot(df_trend['week'], df_trend['sales_ma4'], label='4-week moving avg', linewidth=2)
plt.xlabel('week')
plt.ylabel('sales')
plt.title('Sales with 4-week moving average')
plt.legend()
plt.show()


### 🧪 Experiment: Try Different Window Sizes
Change `window=8` below and observe how the line becomes smoother but loses short-term detail.

**Try these experiments:**
1. First, run with `window=8` (already set)
2. Then try `window=12` - what happens?
3. Try `window=2` - is it too noisy?

**Key Insight:** Larger window = smoother line but more lag; smaller window = more responsive but noisier.

In [ ]:
df_trend['sales_ma8'] = df_trend['sales'].rolling(window=8).mean()
plt.plot(df_trend['week'], df_trend['sales'], label='sales')
plt.plot(df_trend['week'], df_trend['sales_ma8'], label='8-week moving avg', linewidth=2)
plt.xlabel('week')
plt.ylabel('sales')
plt.title('Sales with 8-week moving average')
plt.legend()
plt.show()


### 🧪 Challenge: Compare Multiple Window Sizes

In [ ]:
# TODO: Plot the original sales with THREE different moving averages on the same chart
# This helps you compare how different window sizes affect smoothing

# Step 1: Calculate moving averages for windows 4, 8, and 12
df_trend['ma_4'] = df_trend['sales'].rolling(window=___).mean()   # Fill in
df_trend['ma_8'] = df_trend['sales'].rolling(window=___).mean()   # Fill in  
df_trend['ma_12'] = df_trend['sales'].rolling(window=___).mean()  # Fill in

# Step 2: Plot all lines
plt.figure(figsize=(10, 5))
plt.plot(df_trend['week'], df_trend['sales'], label='Raw Sales', alpha=0.5)
plt.plot(df_trend['week'], df_trend['ma_4'], label='MA-4', linewidth=2)
plt.plot(df_trend['week'], df_trend['ma_8'], label='MA-8', linewidth=2)
plt.plot(df_trend['week'], df_trend['ma_12'], label='MA-12', linewidth=2)
plt.xlabel('Week')
plt.ylabel('Sales')
plt.title('Comparing Moving Average Window Sizes')
plt.legend()
plt.show()

# 🤔 Which window size would you choose for this data? Why?

---
## 4) Quick EDA on a small messy dataset
Combine regression data with a categorical column and some missing values to practice EDA steps:
- `info()`
- missing value counts
- `describe()`
- simple plots


In [ ]:
df = df_reg.copy()
df['sleep_hours'] = rng.normal(7, 1.2, len(df)).clip(3,10)
df.loc[rng.choice(len(df), 5, replace=False), 'sleep_hours'] = np.nan

df['group'] = np.where(df['study_hours'] >= df['study_hours'].median(), 'HighStudy', 'LowStudy')

# EDA outputs
print('Info:')
df.info()

print('\nMissing counts:')
print(df.isna().sum())

print('\nDescribe (numeric):')
print(df.describe())

plt.hist(df['marks'], bins=15)
plt.title('Distribution of marks')
plt.xlabel('marks')
plt.show()

plt.scatter(df['study_hours'], df['marks'])
plt.xlabel('study_hours')
plt.ylabel('marks')
plt.title('study_hours vs marks')
plt.show()


### ✍️ Your EDA Insights
Write two short insights from the EDA above:

**1. Missingness observation:** *(What column has missing values? How many?)*

Your answer: ___

**2. Distribution/relationship observation:** *(What pattern do you see in the histogram or scatter plot?)*

Your answer: ___

---
## 5) Simple Auto-EDA function
This simple function prints shape, dtypes, missing counts, numeric describe, and makes two plots. It uses only basic pandas and matplotlib so students can read it easily.

In [ ]:
def auto_eda(data, hist_col, x_col, y_col):
    print('Shape:', data.shape)
    print('\nDtypes:')
    print(data.dtypes)

    print('\nMissing values:')
    print(data.isna().sum().to_frame('missing_count'))

    print('\nNumeric describe:')
    print(data.select_dtypes(include='number').describe().T)

    # Histogram
    plt.hist(data[hist_col].dropna(), bins=15)
    plt.title(f'Histogram: {hist_col}')
    plt.xlabel(hist_col)
    plt.show()

    # Scatter
    plt.scatter(data[x_col], data[y_col])
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f'Scatter: {x_col} vs {y_col}')
    plt.show()

# Run auto_eda on the messy df
auto_eda(df, hist_col='marks', x_col='study_hours', y_col='marks')


### 🧪 Exercise: Enhance the Auto-EDA Function
Modify `auto_eda` to also print a correlation matrix of numeric columns.

**Hint:** Use `data.select_dtypes(include='number').corr()`

In [ ]:
# TODO: Copy and modify the auto_eda function to include correlation matrix
def auto_eda_enhanced(data, hist_col, x_col, y_col):
    """Enhanced EDA function with correlation matrix."""
    print('Shape:', data.shape)
    print('\nDtypes:')
    print(data.dtypes)

    print('\nMissing values:')
    print(data.isna().sum().to_frame('missing_count'))

    print('\nNumeric describe:')
    print(data.select_dtypes(include='number').describe().T)

    # TODO: Add correlation matrix here
    # Hint: print('\nCorrelation Matrix:')
    # Hint: print(data.select_dtypes(include='number').corr())
    
    # YOUR CODE HERE:
    

    # Histogram
    plt.hist(data[hist_col].dropna(), bins=15)
    plt.title(f'Histogram: {hist_col}')
    plt.xlabel(hist_col)
    plt.show()

    # Scatter
    plt.scatter(data[x_col], data[y_col])
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f'Scatter: {x_col} vs {y_col}')
    plt.show()

# Test your enhanced function:
# auto_eda_enhanced(df, hist_col='marks', x_col='study_hours', y_col='marks')

---
## 6) Practice Questions 📝
A mix of conceptual and coding tasks to consolidate learning.

### Conceptual Questions (Discuss or Write)
Answer these in your own words:

| # | Question | Your Answer |
|---|----------|-------------|
| 1 | In words, what does the regression slope mean? | *(write here)* |
| 2 | Why might the intercept be meaningless sometimes? | *(write here)* |
| 3 | What does R² indicate simply? | *(write here)* |
| 4 | What pattern in residuals suggests a non-linear relationship? | *(write here)* |

### Coding Challenges
Complete the coding tasks in the cells below:

#### Challenge 1: Manual Slope & Intercept Calculation
Compute slope and intercept using the formulas:
- slope m = cov(x,y) / var(x)
- intercept c = mean(y) - m * mean(x)

---

> 💡 **What is `ddof`?**  
> `ddof` = **Delta Degrees of Freedom**
> 
> | `ddof` | Divides by | When to use |
> |--------|------------|-------------|
> | `ddof=0` | n | Population (you have ALL data) |
> | `ddof=1` | n - 1 | **Sample** (subset of population) ← *most common* |

> 🤔 **Why divide by n-1 instead of n?**
> 
> **Simple analogy:** Imagine you have 5 friends and want to guess the average height of ALL students in your school.
> 
> 1. You measure your 5 friends → this is your **sample**
> 2. You calculate their average height → this is your **sample mean**
> 
> **The problem:** Your 5 friends' heights cluster around *their* average, not the *true school average*. So if you calculate how spread out they are (variance), you'll get a number that's **too small**.
> 
> **The fix:** Divide by 4 instead of 5 (n-1 instead of n). This makes the answer slightly bigger to compensate.
> 
> **Think of it this way:** Once you know the average and 4 of the heights, the 5th height is *locked* — you can calculate it. So you really only have **4 free pieces of information**, not 5.
> 
> | Sample size | Big difference? |
> |-------------|-----------------|
> | n = 5 | Yes! (divide by 4 vs 5 = 20% diff) |
> | n = 1000 | Barely (divide by 999 vs 1000 ≈ 0.1% diff) |

In [ ]:
# Challenge 1: Calculate slope and intercept manually
x = df_reg['study_hours'].to_numpy()
y = df_reg['marks'].to_numpy()

# TODO: Calculate covariance of x and y
# Hint: np.cov(x, y, ddof=1)[0,1] gives covariance
cov_xy = ___

# TODO: Calculate variance of x  
# Hint: np.var(x, ddof=1)
var_x = ___

# TODO: Calculate slope
m_manual = ___

# TODO: Calculate intercept
c_manual = ___

print(f'Manual slope m = {m_manual:.3f}')
print(f'Manual intercept c = {c_manual:.3f}')

# Compare with np.polyfit results - they should match!

#### Challenge 2: Find Best and Worst Weeks

In [ ]:
# Challenge 2: Find the week with maximum and minimum sales

# TODO: Find the index of max sales
# Hint: df_trend['sales'].idxmax()
max_idx = ___

# TODO: Get the week number for max sales
max_week = ___

# TODO: Find the week with minimum sales
min_idx = ___
min_week = ___

print(f'Best week (highest sales): Week {max_week}')
print(f'Worst week (lowest sales): Week {min_week}')

# Bonus: What were the actual sales values?
print(f'Max sales: {df_trend.loc[max_idx, "sales"]:.2f}')
print(f'Min sales: {df_trend.loc[min_idx, "sales"]:.2f}')

---
## 🎯 Session Summary & Self-Check

### Key Takeaways
| Concept | What You Learned |
|---------|------------------|
| **Linear Regression** | How to fit a line using `np.polyfit(x, y, 1)` and interpret slope/intercept |
| **R² (R-squared)** | Measures how well the line fits (0 = poor, 1 = perfect) |
| **Residuals** | Difference between actual and predicted; patterns reveal model problems |
| **Moving Average** | Smooths time series data; larger window = smoother but more lag |
| **EDA** | Systematic exploration: shape, dtypes, missing values, describe, plots |

### Self-Assessment Checklist
Before moving on, make sure you can:
- [ ] Explain what slope and intercept mean in a regression
- [ ] Calculate and interpret R²
- [ ] Read a residual plot and identify problems
- [ ] Apply moving average smoothing to time series
- [ ] Perform basic EDA on a new dataset

### 🚀 Next Steps
1. Try these techniques on a real dataset
2. Explore scikit-learn's `LinearRegression` for more advanced features
3. Learn about multiple regression (more than one predictor variable)

---
## 📚 Complete Answer Key (All Exercises)

<details>
<summary><b>🔑 Click to expand all answers</b></summary>

---

### ✅ Setup Quick Check (Cell 4)
**Answer:** Option 1 — Sets the default figure size to 7 inches wide × 4 inches tall

`plt.rcParams` is a dictionary of runtime configuration settings. Setting `figure.figsize` means all future plots will use this size by default.

---

### 🧪 Explore the Data (Cell 11)
```python
# Exercise 1 - Basic statistics
df_reg.describe()

# Exercise 2 - Correlation
df_reg['study_hours'].corr(df_reg['marks'])  # Should be ~0.8-0.9 (strong positive)

# Exercise 3 - Number of weeks
len(df_trend)  # Answer: 52 weeks
# OR: df_trend.shape[0]
```

---

### 🧪 Predict Marks (Cell 20)
```python
# For 3 hours of study:
study_hours_new = 3
predicted_marks = m * study_hours_new + c

# For 8 hours of study:
study_hours_new = 8
predicted_marks = m * study_hours_new + c

# For 0 hours (intercept interpretation):
study_hours_new = 0
predicted_marks = m * 0 + c  # This equals c (the intercept)
# The result is the intercept (~20), which represents base marks with zero study
```

---

### ✍️ Interpret the Slope (Cell 24)
**The slope means:** For each additional hour of study, a student's marks increase by approximately 12 points on average.

**One limitation:** This assumes a linear relationship; in reality, there may be diminishing returns (studying 20 hours isn't necessarily twice as effective as 10 hours). Also, correlation ≠ causation.

---

### 🤔 Trend Observation Questions (Cell 27)
1. **Trend:** Yes, there's an overall **upward trend** (sales increase over the year)
2. **Seasonality:** Yes, there's a **wave-like pattern** repeating roughly every 12 weeks
3. **Unusual points:** Some random spikes/dips due to noise, but no extreme outliers

---

### 🧪 Moving Average Challenge (Cell 33)
```python
df_trend['ma_4'] = df_trend['sales'].rolling(window=4).mean()
df_trend['ma_8'] = df_trend['sales'].rolling(window=8).mean()
df_trend['ma_12'] = df_trend['sales'].rolling(window=12).mean()
```

**Which to choose?** MA-8 or MA-12 would be good choices — they smooth out noise while still showing the trend. MA-4 may be too noisy, while larger windows lose too much detail.

---

### ✍️ EDA Insights (Cell 36)
**1. Missingness:** The `sleep_hours` column has **5 missing values** (NaN). All other columns have 0 missing.

**2. Distribution/relationship:** 
- The histogram shows marks are roughly **normally distributed**, centered around 60-70
- The scatter plot shows a **positive linear relationship** between study_hours and marks

---

### 🧪 Auto-EDA Enhancement (Cell 40)
```python
def auto_eda_enhanced(data, hist_col, x_col, y_col):
    """Enhanced EDA function with correlation matrix."""
    print('Shape:', data.shape)
    print('\nDtypes:')
    print(data.dtypes)
    print('\nMissing values:')
    print(data.isna().sum().to_frame('missing_count'))
    print('\nNumeric describe:')
    print(data.select_dtypes(include='number').describe().T)
    
    # ✅ ADD THIS:
    print('\nCorrelation Matrix:')
    print(data.select_dtypes(include='number').corr())
    
    # ... rest of function (histogram and scatter)
```

---

### 📝 Conceptual Questions (Cell 41)

| # | Question | Answer |
|---|----------|--------|
| 1 | What does the regression slope mean? | The slope is the average change in Y for each 1-unit increase in X. Here: ~12 more marks per extra study hour. |
| 2 | Why might intercept be meaningless? | When X=0 is outside the data range or impossible (e.g., negative study hours). It's just where the line crosses Y-axis. |
| 3 | What does R² indicate? | The proportion of variance in Y explained by X. R²=0.8 means 80% of mark variation is explained by study hours. |
| 4 | What residual pattern suggests non-linearity? | A **curved/U-shaped pattern** in residuals indicates the true relationship is non-linear (try polynomial regression). |

---

### Challenge 1: Manual Slope & Intercept (Cell 43)
```python
cov_xy = np.cov(x, y, ddof=1)[0,1]  # Covariance of x and y
var_x = np.var(x, ddof=1)           # Variance of x
m_manual = cov_xy / var_x           # Slope formula
c_manual = y.mean() - m_manual * x.mean()  # Intercept formula
```

---

### Challenge 2: Best and Worst Weeks (Cell 45)
```python
max_idx = df_trend['sales'].idxmax()
max_week = int(df_trend.loc[max_idx, 'week'])

min_idx = df_trend['sales'].idxmin()
min_week = int(df_trend.loc[min_idx, 'week'])
```

---

</details>